# Graph Calculus Hackathon

In this hackathon you will build core graph-calculus operators from scratch and apply them to three problems spanning spectral theory, machine learning, and data analysis.

**Outline:**

| Part | Topic | Key Ideas |
|------|-------|-----------|
| 1 | **Graph Laplacian from Exterior Calculus** | Coboundary $\delta_0$, codifferential $\delta_0^*$, eigenvalues, Fiedler vector |
| 2 | **Graph Attention Network** | Learned codifferential, node classification on Cora |
| 3 | **Ranking via Hodge Decomposition** | Pairwise preferences as edge flows, acyclic ranking, curl = intransitive cycles |

**Prerequisites:** Lectures 16–17 (graph calculus, Hodge decomposition).  
**Libraries:** NumPy, SciPy, NetworkX, Matplotlib, PyTorch, PyTorch Geometric.

## 0. Setup

In [ ]:
# Uncomment the lines below if running in Google Colab
# !pip install -q torch-scatter torch-sparse torch-geometric networkx

In [ ]:
import numpy as np
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
from torch_geometric.utils import to_networkx

---
## Part 1 — Graph Laplacian from Exterior Calculus

Recall from Lecture 16 the discrete exterior calculus on a graph $G = (V, E)$:

| Operator | Symbol | Continuous analogue | Action on edge $e = (i,j)$ |
|----------|--------|--------------------|--------------------------|
| Coboundary | $\delta_0$ | Gradient $\nabla$ | $(\delta_0 \phi)_e = \phi_j - \phi_i$ |
| Codifferential | $\delta_0^*$ | Divergence $-\nabla\cdot$ | $(\delta_0^* f)_i = \sum_{j\sim i} w_{ij} f_{ij}$ |
| Laplacian | $L = \delta_0^T W \delta_0$ | $-\Delta$ | Weighted graph Laplacian |

The Laplacian's spectrum controls everything: connectivity ($\lambda_0 = 0$), coercivity ($\lambda_1 =$ Fiedler eigenvalue), and smoothing rates.

### Helper: Create standard graphs

In [ ]:
def make_graph(graph_type, n=20):
    """Create a standard graph.
    
    Parameters
    ----------
    graph_type : str — 'path', 'cycle', 'grid', or 'petersen'
    n : int — nodes (path/cycle) or nodes per side (grid); ignored for petersen
    """
    if graph_type == 'path':
        G = nx.path_graph(n)
    elif graph_type == 'cycle':
        G = nx.cycle_graph(n)
    elif graph_type == 'grid':
        G = nx.grid_2d_graph(n, n)
        G = nx.convert_node_labels_to_integers(G)
    elif graph_type == 'petersen':
        G = nx.petersen_graph()
    else:
        raise ValueError(f"Unknown graph type: {graph_type}")
    return G

# Quick visualization
G = make_graph('path', 20)
pos = nx.spring_layout(G, seed=42)
fig, ax = plt.subplots(figsize=(8, 3))
nx.draw(G, pos, ax=ax, with_labels=True, node_color='lightblue',
        node_size=300, font_size=8, edge_color='gray')
ax.set_title(f"Path graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
plt.tight_layout()
plt.show()

### TODO 1 — Build the coboundary operator $\delta_0$

The coboundary (graph gradient) is the signed incidence matrix. For each edge $e = (i, j)$ with $i < j$:

$$\delta_0[e, i] = -1, \quad \delta_0[e, j] = +1$$

so that $(\delta_0 \phi)_e = \phi_j - \phi_i$.

**Implement** `build_coboundary(G)` returning a sparse $|E| \times N$ matrix and the sorted edge list.

*Hint:* Use `sp.csr_matrix((vals, (rows, cols)), shape=...)` to build the sparse matrix.

In [ ]:
def build_coboundary(G):
    """Build the coboundary operator δ₀ (graph gradient).
    
    Returns
    -------
    delta0 : scipy.sparse.csr_matrix, shape (|E|, N)
    edges  : list of (i, j) with i < j, sorted
    """
    N = G.number_of_nodes()
    edges = sorted(set((min(u, v), max(u, v)) for u, v in G.edges()))
    E = len(edges)

    rows, cols, vals = [], [], []
    # ----- TODO: fill in the COO entries for the sparse matrix -----
    # For each edge index idx and edge (i, j):
    #   row idx, col i → val -1
    #   row idx, col j → val +1
    ...
    # ---------------------------------------------------------------

    delta0 = sp.csr_matrix((vals, (rows, cols)), shape=(E, N))
    return delta0, edges

# --- verify ---
delta0, edges = build_coboundary(G)
print(f"δ₀ shape: {delta0.shape}  (|E|×N = {len(edges)}×{G.number_of_nodes()})")
print(f"Non-zeros: {delta0.nnz}  (should be 2·|E| = {2*len(edges)})")
# δ₀ applied to a constant should be zero
print(f"δ₀ · 1 = {np.abs(delta0 @ np.ones(G.number_of_nodes())).max():.0e}  (should be 0)")

### TODO 2 — Build the weighted graph Laplacian

$$L = \delta_0^T\, W\, \delta_0$$

where $W = \text{diag}(w_e)$ is a diagonal edge-weight matrix. For uniform weights $W = I$.

**Implement** `build_laplacian(delta0, weights=None)`.

*Hint:* Use `sp.diags(weights)` to create $W$.

In [ ]:
def build_laplacian(delta0, weights=None):
    """Build L = δ₀ᵀ W δ₀.
    
    Parameters
    ----------
    delta0  : sparse (|E|, N)
    weights : array of length |E|, or None for uniform
    """
    # ----- TODO: compute L = δ₀ᵀ W δ₀ -----
    ...
    # ----------------------------------------
    return L

# --- verify ---
L = build_laplacian(delta0)
print(f"L shape: {L.shape}")
print(f"Symmetric: {np.allclose(L.toarray(), L.toarray().T)}")
print(f"Max row-sum: {np.abs(L @ np.ones(L.shape[0])).max():.0e}  (should be 0)")

### TODO 3 — Eigenvalue analysis

The spectrum of $L$ encodes graph structure:

- $\lambda_0 = 0$ with eigenvector $\mathbf{1}$ (connected graph)
- $\lambda_1$ = **Fiedler eigenvalue** = coercivity constant (spectral gap)
- **Diameter bound**: $\lambda_1 \geq \frac{1}{D \cdot \text{vol}(G)}$ where $D$ is the diameter and $\text{vol}(G) = \sum_i d_i$
- For a path graph: $\lambda_k = 2\bigl(1 - \cos(k\pi/n)\bigr)$

**Tasks:**
1. Compute the full spectrum of the path graph ($n=20$) using `np.linalg.eigvalsh`
2. Verify $\lambda_0 \approx 0$ and compare $\lambda_1$ with both the analytic value and the diameter bound
3. Plot the spectrum with the bound shown as a horizontal line

In [ ]:
n = 20
G_path = make_graph('path', n)
d0_path, _ = build_coboundary(G_path)
L_path = build_laplacian(d0_path)

# ----- TODO: compute eigenvalues, analytic value, diameter bound -----
# eigenvalues = ...
# lambda1_exact = ...
# D = ...
# vol = ...
# bound = ...
# print results and create a stem plot
...
# --------------------------------------------------------------------

**TODO 3b** — Repeat for cycle, grid (5×5), and Petersen graphs. Create a 2×2 panel comparing the four spectra.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
specs = [('path', 20, 'Path (n=20)'),
         ('cycle', 20, 'Cycle (n=20)'),
         ('grid', 5, '5×5 Grid'),
         ('petersen', None, 'Petersen')]

for ax, (gtype, p, title) in zip(axes.flat, specs):
    Gt = make_graph(gtype, p) if p else make_graph(gtype)
    # ----- TODO: build operators, compute eigenvalues, plot -----
    ...
    # ------------------------------------------------------------

plt.tight_layout(); plt.show()

**Discussion:** How does the spectral gap $\lambda_1$ change with connectivity? Which graph has the tightest diameter bound? How does the Petersen graph (a strongly regular graph) differ from the lattice graphs?

### Fiedler eigenvector visualization

The Fiedler eigenvector (eigenvector for $\lambda_1$) gives the optimal graph bipartition — nodes are colored by their sign/magnitude.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
vis_specs = [('path', 20, 'Path'), ('cycle', 20, 'Cycle'), ('grid', 5, '5×5 Grid')]

for ax, (gtype, p, title) in zip(axes, vis_specs):
    Gv = make_graph(gtype, p)
    d0v, _ = build_coboundary(Gv)
    Lv = build_laplacian(d0v)
    evals, evecs = np.linalg.eigh(Lv.toarray())
    fiedler = evecs[:, 1]

    if gtype == 'grid':
        pos_v = {i: (i % p, i // p) for i in range(Gv.number_of_nodes())}
    else:
        pos_v = nx.spring_layout(Gv, seed=42)

    nodes = nx.draw_networkx_nodes(Gv, pos_v, ax=ax, node_color=fiedler,
                                   cmap=cm.RdBu, node_size=200)
    nx.draw_networkx_edges(Gv, pos_v, ax=ax, edge_color='gray', alpha=0.4)
    plt.colorbar(nodes, ax=ax, shrink=0.8)
    ax.set_title(f'{title} — Fiedler eigenvector'); ax.axis('off')

plt.tight_layout(); plt.show()

---
## Part 2 — Graph Attention Network

From Lecture 16, a Graph Attention Network (GAT) layer performs message passing:

$$x_i^{n+1} = x_i^n + \sigma\!\left(\sum_{j \sim i} \alpha_{ij}\, W x_j^n\right)$$

where $\alpha_{ij}$ are learned attention weights on edges. In our exterior calculus language, this is:

$$x^{n+1} = x^n + \sigma\bigl(\delta_0^{*,\alpha}\, m\bigr)$$

— an **attention-weighted codifferential** aggregating edge messages to nodes.

### Dataset: Cora

Cora is a citation network: 2,708 papers (nodes), 10,556 citations (edges), 1,433-dim bag-of-words features, 7 topic classes. The task is **transductive node classification** — predict paper topics from features + graph structure.

In [ ]:
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]
print(f"Nodes: {data.num_nodes}  Edges: {data.num_edges}")
print(f"Features: {data.num_node_features}  Classes: {dataset.num_classes}")
print(f"Train/Val/Test: {data.train_mask.sum()}/{data.val_mask.sum()}/{data.test_mask.sum()}")

In [ ]:
# Visualize a subgraph
G_cora = to_networkx(data, to_undirected=True)
sub_nodes = sorted(nx.single_source_shortest_path_length(G_cora, 0, cutoff=2).keys())[:50]
G_sub = G_cora.subgraph(sub_nodes)

fig, ax = plt.subplots(figsize=(8, 6))
pos_sub = nx.spring_layout(G_sub, seed=42)
nx.draw(G_sub, pos_sub, ax=ax, node_size=80,
        node_color=[data.y[n].item() for n in G_sub.nodes()],
        cmap=cm.Set3, edge_color='gray', alpha=0.8)
ax.set_title('Cora subgraph (colored by class)'); plt.show()

### TODO 4 — Build a GAT classifier

Use `GATConv` from PyTorch Geometric.

**Architecture:**
```
GATConv(1433, 8, heads=8)  →  ELU  →  Dropout(0.6)
GATConv(64, 7, heads=1)    →  log_softmax
```

**Implement** the `__init__` and `forward` methods.

*Hint:* 
- `GATConv(in_ch, out_ch, heads=H)` outputs dimension `out_ch * H` (default `concat=True`)
- For the final layer, use `concat=False` so the output has dimension `out_ch`

In [ ]:
class GraphAttentionClassifier(nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch, heads=8, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        # ----- TODO: define self.conv1 and self.conv2 -----
        # self.conv1 = GATConv(...)
        # self.conv2 = GATConv(..., concat=False)
        ...
        # --------------------------------------------------

    def forward(self, x, edge_index):
        # ----- TODO: implement forward pass -----
        # Dropout → conv1 → ELU → Dropout → conv2 → log_softmax
        ...
        # ----------------------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GraphAttentionClassifier(
    in_ch=dataset.num_node_features,
    hid_ch=8, out_ch=dataset.num_classes
).to(device)
data = data.to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

### TODO 5 — Train and evaluate

- Optimizer: Adam, `lr=0.005`, `weight_decay=5e-4`
- Loss: `F.nll_loss` on training mask nodes
- 200 epochs, print every 20

*Hint:* The `evaluate()` function is provided. You need to implement `train()` and the training loop.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

def train():
    # ----- TODO: implement one training step -----
    # model.train(), zero_grad, forward, compute loss on train_mask, backward, step
    ...
    # return loss.item()
    # ----------------------------------------------

@torch.no_grad()
def evaluate():
    model.eval()
    pred = model(data.x, data.edge_index).argmax(dim=1)
    accs = {}
    for name, mask in [('train', data.train_mask),
                       ('val', data.val_mask),
                       ('test', data.test_mask)]:
        accs[name] = (pred[mask] == data.y[mask]).float().mean().item()
    return accs

# ----- TODO: training loop — 200 epochs, print every 20 -----
...
# -------------------------------------------------------------

accs = evaluate()
print(f"\nFinal — Train: {accs['train']:.3f}  Val: {accs['val']:.3f}  Test: {accs['test']:.3f}")

**Discussion:** How does the GAT compare to a simple MLP that ignores graph structure? (An MLP on Cora typically gets ~55–60% test accuracy.) What role does the learned codifferential play?

### Attention weight visualization

We can inspect what the learned codifferential $\delta_0^{*,\alpha}$ looks like by extracting the attention coefficients $\alpha_{ij}$ and plotting edge widths proportional to attention.

In [ ]:
model.eval()
with torch.no_grad():
    _, (ei_att, alpha_att) = model.conv1(
        data.x, data.edge_index, return_attention_weights=True)

alpha_mean = alpha_att.mean(dim=1).cpu().numpy()
ei_np = ei_att.cpu().numpy()

# Filter to our subgraph
sub_set = set(sub_nodes)
mask = [(ei_np[0, k] in sub_set and ei_np[1, k] in sub_set)
        for k in range(ei_np.shape[1])]
mask = np.array(mask)

fig, ax = plt.subplots(figsize=(8, 6))
nx.draw_networkx_nodes(G_sub, pos_sub, ax=ax, node_size=80,
                       node_color=[data.y[n].item() for n in G_sub.nodes()],
                       cmap=cm.Set3)
if mask.any():
    se = ei_np[:, mask]; sa = alpha_mean[mask]
    elist = [(se[0, k], se[1, k]) for k in range(se.shape[1])
             if G_sub.has_edge(se[0, k], se[1, k])]
    ea = [sa[k] for k in range(se.shape[1])
          if G_sub.has_edge(se[0, k], se[1, k])]
    if elist:
        w = np.array(ea); w = 1 + 5*(w - w.min())/(w.max() - w.min() + 1e-8)
        nx.draw_networkx_edges(G_sub, pos_sub, ax=ax, edgelist=elist,
                               width=w, edge_color='steelblue', alpha=0.7)
ax.set_title('Learned attention weights (width ∝ α)'); ax.axis('off')
plt.tight_layout(); plt.show()

---
## Part 3 — Ranking via Hodge Decomposition

Given **pairwise comparison data** $Y_{ij}$ ("how much is item $j$ preferred over item $i$"), we want a **global ranking** $s$ such that $s_j - s_i \approx Y_{ij}$.

From Lecture 17, the **graph Hodge decomposition** splits any edge flow:

$$Y = \underbrace{\delta_0 s}_{\text{gradient (ranking)}} + \underbrace{\delta_1^T A}_{\text{curl (cycles)}} + \underbrace{h}_{\text{harmonic}}$$

- **Gradient component** $\delta_0 s$: the part consistent with a global ranking (acyclic)
- **Curl component** $\delta_1^T A$: intransitive cycles (rock-paper-scissors patterns)
- **Harmonic component** $h$: global structure in the kernel of both $\delta_0^T$ and $\delta_1$

### Synthetic comparison data

In [ ]:
def generate_ranking_data(N=10, noise_std=0.3, n_cycles=2, cycle_strength=2.0, seed=42):
    """Synthetic pairwise comparisons with injected intransitive cycles."""
    rng = np.random.RandomState(seed)
    s_true = rng.randn(N)
    s_true -= s_true.mean()
    G = nx.complete_graph(N)

    # Pairwise comparisons Y_{ij} = s*_j - s*_i + noise
    Y = {}
    for u, v in G.edges():
        i, j = min(u, v), max(u, v)
        Y[(i, j)] = (s_true[j] - s_true[i]) + noise_std * rng.randn()

    # Inject intransitive cycles on random triangles
    tris = [(i, j, k) for i in range(N) for j in range(i+1, N) for k in range(j+1, N)]
    chosen = rng.choice(len(tris), size=min(n_cycles, len(tris)), replace=False)
    for idx in chosen:
        i, j, k = tris[idx]
        Y[(i, j)] += cycle_strength
        Y[(j, k)] += cycle_strength
        Y[(i, k)] -= cycle_strength  # reverses the (i,k) edge → creates circulation
        print(f"  Injected cycle in triangle ({i},{j},{k})")

    return G, Y, s_true

G_rank, Y_data, s_true = generate_ranking_data(N=10)
print(f"\n{G_rank.number_of_nodes()} items, {G_rank.number_of_edges()} comparisons")
print(f"True ranking (best→worst): {np.argsort(-s_true)}")

### TODO 6 — Solve for the global ranking

Minimizing $\sum_{(i,j)} (Y_{ij} - (\delta_0 s)_{ij})^2$ leads to the **normal equations**:

$$\delta_0^T \delta_0\, s \;=\; \delta_0^T Y$$

This is exactly the **graph Laplacian system** from Part 1! Pin $s_0 = 0$ to remove the constant null space.

**Tasks:**
1. Reuse `build_coboundary` to get $\delta_0$ for the ranking graph
2. Build $L = \delta_0^T \delta_0$ and the right-hand side $\delta_0^T Y$
3. Remove row/column 0 (pinning $s_0 = 0$) and solve
4. Compare recovered ranking with ground truth

In [ ]:
delta0_r, edges_r = build_coboundary(G_rank)
N_r = G_rank.number_of_nodes()
Y_vec = np.array([Y_data[e] for e in edges_r])

# ----- TODO: assemble normal equations and solve -----
# L_r = ...
# rhs = ...
# Pin node 0: remove row/col 0 from L and entry 0 from rhs
# L_pin = ...
# rhs_pin = ...
# s_rec = np.zeros(N_r)
# s_rec[1:] = np.linalg.solve(...)
# s_rec -= s_rec.mean()
...
# -----------------------------------------------------

s_true_c = s_true - s_true.mean()
corr = np.corrcoef(s_rec, s_true_c)[0, 1]
print(f"Correlation with ground truth: {corr:.4f}")
print(f"Recovered order: {np.argsort(-s_rec)}")
print(f"True order:      {np.argsort(-s_true_c)}")

In [ ]:
# Visualize ranking recovery
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(s_true_c, s_rec, s=100, c='steelblue', edgecolors='k', zorder=3)
for i in range(N_r):
    ax.annotate(f' {i}', (s_true_c[i], s_rec[i]), fontsize=9)
lims = [min(s_true_c.min(), s_rec.min())-0.3, max(s_true_c.max(), s_rec.max())+0.3]
ax.plot(lims, lims, 'r--', alpha=0.5, label='Perfect')
ax.set_xlabel('True score'); ax.set_ylabel('Recovered score')
ax.set_title(f'Ranking Recovery (ρ = {corr:.3f})'); ax.legend()
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

ax = axes[1]
x_pos = np.arange(N_r); w = 0.35
ax.bar(x_pos - w/2, s_true_c, w, label='True', color='steelblue', alpha=0.7)
ax.bar(x_pos + w/2, s_rec, w, label='Recovered', color='coral', alpha=0.7)
ax.set_xlabel('Item'); ax.set_ylabel('Score')
ax.set_title('Scores by Item'); ax.legend(); ax.set_xticks(x_pos)

plt.tight_layout(); plt.show()

### TODO 7 — Hodge decomposition: consistent vs. cyclic preferences

Now we separate the edge flow $Y$ into its three orthogonal components.

**Step 1:** Build the **curl operator** $\delta_1$ from triangles. For triangle $(i,j,k)$ with $i<j<k$:

$$\delta_1[t,\, e_{ij}] = +1, \quad \delta_1[t,\, e_{jk}] = +1, \quad \delta_1[t,\, e_{ik}] = -1$$

The sign on $e_{ik}$ is $-1$ because the cycle orientation $(i{\to}j{\to}k{\to}i)$ traverses edge $(i,k)$ backwards.

**Step 2:** Solve $\delta_1 \delta_1^T A = \delta_1(Y - \delta_0 s)$ for the curl coefficients.

**Step 3:** Compute: gradient = $\delta_0 s$, curl = $\delta_1^T A$, harmonic = $Y - \text{grad} - \text{curl}$.

*Hint:* `find_triangles(G)` is provided below. Use `np.linalg.lstsq` for the curl system (it may be singular).

In [ ]:
def find_triangles(G):
    """Return all triangles (i,j,k) with i < j < k."""
    adj = {n: set(G.neighbors(n)) for n in G.nodes()}
    tris = []
    for i in sorted(G.nodes()):
        for j in sorted(adj[i]):
            if j > i:
                for k in sorted(adj[i] & adj[j]):
                    if k > j:
                        tris.append((i, j, k))
    return tris

def build_curl_operator(G, edges):
    """Build δ₁ mapping 1-cochains to 2-cochains.
    
    Returns
    -------
    delta1    : sparse (|T|, |E|)
    triangles : list of (i, j, k)
    """
    triangles = find_triangles(G)
    edge_idx = {e: k for k, e in enumerate(edges)}
    E = len(edges)
    T = len(triangles)
    rows, cols, vals = [], [], []

    # ----- TODO: fill COO entries for δ₁ -----
    # For each triangle t = (i,j,k):
    #   (t, edge_idx[(i,j)]) → +1
    #   (t, edge_idx[(j,k)]) → +1
    #   (t, edge_idx[(i,k)]) → -1
    ...
    # ------------------------------------------

    delta1 = sp.csr_matrix((vals, (rows, cols)), shape=(T, E))
    return delta1, triangles

delta1, tris = build_curl_operator(G_rank, edges_r)
print(f"δ₁ shape: {delta1.shape}  ({len(tris)} triangles × {len(edges_r)} edges)")
print(f"‖δ₁ δ₀‖ = {sp.linalg.norm(delta1 @ delta0_r):.2e}  (curl∘grad = 0 ✓)")

In [ ]:
# ----- TODO: compute the Hodge decomposition -----
# grad_comp = δ₀ @ s_rec
# residual  = Y_vec - grad_comp
# Solve δ₁ δ₁ᵀ A = δ₁ residual  (use np.linalg.lstsq)
# curl_comp = δ₁ᵀ @ A
# harm_comp = Y_vec - grad_comp - curl_comp
...
# -------------------------------------------------

# Report norms
nY  = np.linalg.norm(Y_vec)**2
nG  = np.linalg.norm(grad_comp)**2
nC  = np.linalg.norm(curl_comp)**2
nH  = np.linalg.norm(harm_comp)**2

print("Hodge Decomposition: Y = δ₀s + δ₁ᵀA + h")
print(f"  ‖Y‖²    = {nY:.4f}")
print(f"  ‖δ₀s‖²  = {nG:.4f}  ({100*nG/nY:.1f}%)")
print(f"  ‖δ₁ᵀA‖² = {nC:.4f}  ({100*nC/nY:.1f}%)")
print(f"  ‖h‖²    = {nH:.4f}  ({100*nH/nY:.1f}%)")
print(f"  Sum      = {nG+nC+nH:.4f}")
print(f"  Pythagorean identity: {np.isclose(nY, nG+nC+nH, rtol=1e-6)}")

**Discussion:** What fraction of the data is explained by a consistent ranking? Where do the cyclic components concentrate? Do they correspond to the injected intransitive triangles?

In [ ]:
# Visualize the three components on the graph
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
comps = [(grad_comp, 'Gradient δ₀s\n(consistent ranking)', 'Blues'),
         (curl_comp, 'Curl δ₁ᵀA\n(intransitive cycles)', 'Reds'),
         (harm_comp, 'Harmonic h', 'Greens')]
pos_r = nx.spring_layout(G_rank, seed=42)

for ax, (comp, title, cmap_name) in zip(axes, comps):
    nx.draw_networkx_nodes(G_rank, pos_r, ax=ax, node_color='lightgray',
                           node_size=400, edgecolors='black')
    nx.draw_networkx_labels(G_rank, pos_r, ax=ax, font_size=10)
    ec = np.abs(comp)
    if ec.max() > 0:
        ec = ec / ec.max()
    nx.draw_networkx_edges(G_rank, pos_r, ax=ax,
                           edgelist=list(edges_r), edge_color=ec,
                           edge_cmap=plt.get_cmap(cmap_name),
                           width=2, edge_vmin=0, edge_vmax=1)
    ax.set_title(title); ax.axis('off')

plt.tight_layout(); plt.show()

---
## Bonus Exercises

**A.** Apply the ranking algorithm to real data: download a small sports tournament bracket (e.g., NCAA results) or a MovieLens ratings subset, construct the comparison graph, and extract the Hodge decomposition. Which matchups are most "intransitive"?

**B.** Compare the GAT's learned node embeddings with the Fiedler eigenvector from Part 1. Compute the embedding of each Cora node from the GAT's first hidden layer, project to 2D via PCA, and color by class. Does the GAT implicitly learn spectral features?

**C.** Add DEC Hodge star weights to Part 1: on a path graph representing $[0, L]$ with spacing $h$, set $w_e = (\star_1)_e = 1/h$. Recompute the eigenvalues of the **generalized** eigenvalue problem $K u = \lambda M u$ where $K = \delta_0^T W \delta_0$ and $M = \text{diag}(h)$. Verify that $\lambda_1 \to \pi^2/L^2$ as $h \to 0$, following the coercivity proof from Lecture 16.

---
*Graph Calculus Hackathon — ENM 5320, Spring 2026*